In [0]:
customers_df = spark.table(
    "ecommerce_project.project.bronze_customers"
)

products_df = spark.table(
    "ecommerce_project.project.bronze_products"
)

orders_df = spark.table(
    "ecommerce_project.project.bronze_orders"
)

payments_df = spark.table(
    "ecommerce_project.project.bronze_payments"
)

deliveries_df = spark.table(
    "ecommerce_project.project.bronze_deliveries"
)

In [0]:
from pyspark.sql import functions as F

In [0]:
silver_customers_df = customers_df \
    .withColumn("customer_id", F.trim(F.col("customer_id"))) \
    .withColumn("customer_name", F.trim(F.col("customer_name"))) \
    .withColumn("email", F.lower(F.trim(F.col("email")))) \
    .withColumn("city", F.trim(F.col("city"))) \
    .withColumn("state", F.trim(F.col("state"))) \
    .dropDuplicates(["customer_id"])

In [0]:
silver_products_df = products_df \
    .withColumn("product_id", F.trim(F.col("product_id"))) \
    .withColumn("product_name", F.trim(F.col("product_name"))) \
    .withColumn("category", F.trim(F.col("category"))) \
    .filter(
        (F.col("unit_price").isNotNull()) &
        (F.col("unit_price") > 0)
    ) \
    .dropDuplicates(["product_id"])

In [0]:

silver_orders_df = orders_df \
    .withColumn("order_id", F.trim(F.col("order_id"))) \
    .withColumn("customer_id", F.trim(F.col("customer_id"))) \
    .withColumn("product_id", F.trim(F.col("product_id"))) \
    .withColumn(
        "order_date",
        F.expr("try_cast(order_date AS DATE)")
    ) \
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    ) \
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    ) \
    .filter(
        (F.col("quantity").isNotNull()) &
        (F.col("quantity") > 0)
    ) \
    .dropDuplicates(["order_id"])

In [0]:
silver_payments_df = payments_df \
    .withColumn("payment_id", F.trim(F.col("payment_id"))) \
    .withColumn("order_id", F.trim(F.col("order_id"))) \
    .withColumn(
        "payment_method",
        F.lower(F.trim(F.col("payment_method")))
    ) \
    .withColumn(
        "payment_status",
        F.lower(F.trim(F.col("payment_status")))
    ) \
    .filter(
        (F.col("amount").isNotNull()) &
        (F.col("amount") >= 0)
    ) \
    .dropDuplicates(["payment_id"])

In [0]:
silver_deliveries_df = deliveries_df \
    .withColumn("delivery_id", F.trim(F.col("delivery_id"))) \
    .withColumn("order_id", F.trim(F.col("order_id"))) \
    .withColumn(
        "delivery_status",
        F.lower(F.trim(F.col("delivery_status")))
    ) \
    .withColumn("delivery_city", F.trim(F.col("delivery_city"))) \
    .dropDuplicates(["delivery_id"])

In [0]:
# customers
silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_customers"
    )
# Products
silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_products"
    )
# Orders
silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_orders"
    )
# Payments
silver_payments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_payments"
    )
# Deliveries
silver_deliveries_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_deliveries"
    )

In [0]:
%sql
SHOW TABLES IN ecommerce_project.project;

In [0]:
silver_customers_df = spark.table(
    "ecommerce_project.project.silver_customers"
)

silver_products_df = spark.table(
    "ecommerce_project.project.silver_products"
)

silver_orders_df = spark.table(
    "ecommerce_project.project.silver_orders"
)

silver_payments_df = spark.table(
    "ecommerce_project.project.silver_payments"
)

silver_deliveries_df = spark.table(
    "ecommerce_project.project.silver_deliveries"
)

In [0]:
print("Null Customer IDs:",
      silver_customers_df.filter(
          F.col("customer_id").isNull()
      ).count())

print("Null Product IDs:",
      silver_products_df.filter(
          F.col("product_id").isNull()
      ).count())

print("Null Order IDs:",
      silver_orders_df.filter(
          F.col("order_id").isNull()
      ).count())

print("Null Payment IDs:",
      silver_payments_df.filter(
          F.col("payment_id").isNull()
      ).count())

print("Null Delivery IDs:",
      silver_deliveries_df.filter(
          F.col("delivery_id").isNull()
      ).count())

In [0]:
print("Duplicate Customers:",
      silver_customers_df.groupBy("customer_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Duplicate Products:",
      silver_products_df.groupBy("product_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Duplicate Orders:",
      silver_orders_df.groupBy("order_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Duplicate Payments:",
      silver_payments_df.groupBy("payment_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Duplicate Deliveries:",
      silver_deliveries_df.groupBy("delivery_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

In [0]:
print(
    "Invalid product prices:",
    silver_products_df.filter(
        (F.col("unit_price").isNull()) |
        (F.col("unit_price") <= 0)
    ).count()
)

print(
    "Invalid quantities:",
    silver_orders_df.filter(
        (F.col("quantity").isNull()) |
        (F.col("quantity") <= 0)
    ).count()
)

print(
    "Invalid payment amounts:",
    silver_payments_df.filter(
        (F.col("amount").isNull()) |
        (F.col("amount") < 0)
    ).count()
)

In [0]:
invalid_order_dates = silver_orders_df.filter(
    F.col("order_date").isNull()
)

print(
    "Orders with invalid/missing order_date:",
    invalid_order_dates.count()
)

In [0]:
orphan_orders_customers = silver_orders_df \
    .join(
        silver_customers_df,
        on="customer_id",
        how="left_anti"
    )

print(
    "Orders with invalid customer_id:",
    orphan_orders_customers.count()
)

orphan_orders_products = silver_orders_df \
    .join(
        silver_products_df,
        on="product_id",
        how="left_anti"
    )

print(
    "Orders with invalid product_id:",
    orphan_orders_products.count()
)

In [0]:
orphan_payments = silver_payments_df \
    .join(
        silver_orders_df,
        on="order_id",
        how="left_anti"
    )

print(
    "Payments with invalid order_id:",
    orphan_payments.count()
)

In [0]:
orphan_deliveries = silver_deliveries_df \
    .join(
        silver_orders_df,
        on="order_id",
        how="left_anti"
    )

print(
    "Deliveries with invalid order_id:",
    orphan_deliveries.count()
)

In [0]:
display(orphan_orders_customers)

display(orphan_orders_products)

display(orphan_payments)

display(orphan_deliveries)

display(invalid_order_dates)


In [0]:
from pyspark.sql import functions as F

# Valid reference IDs
valid_customers = silver_customers_df.select("customer_id").distinct()
valid_products = silver_products_df.select("product_id").distinct()

# Identify invalid customer references
invalid_customer_orders = (
    silver_orders_df
    .join(valid_customers, on="customer_id", how="left_anti")
    .withColumn(
        "dq_reason",
        F.when(
            F.col("customer_id").isNull(),
            F.lit("MISSING_CUSTOMER_ID")
        ).otherwise(
            F.lit("INVALID_CUSTOMER_ID")
        )
    )
)

# Identify invalid product references
invalid_product_orders = (
    silver_orders_df
    .join(valid_products, on="product_id", how="left_anti")
    .withColumn(
        "dq_reason",
        F.when(
            F.col("product_id").isNull(),
            F.lit("MISSING_PRODUCT_ID")
        ).otherwise(
            F.lit("INVALID_PRODUCT_ID")
        )
    )
)

# Identify invalid dates
invalid_date_orders = (
    silver_orders_df
    .filter(F.col("order_date").isNull())
    .withColumn(
        "dq_reason",
        F.lit("INVALID_ORDER_DATE")
    )
)

In [0]:
order_dq_records = (
    invalid_customer_orders
    .select("order_id", "dq_reason")
    .union(
        invalid_product_orders
        .select("order_id", "dq_reason")
    )
    .union(
        invalid_date_orders
        .select("order_id", "dq_reason")
    )
    .groupBy("order_id")
    .agg(
        F.concat_ws(
            ", ",
            F.collect_set("dq_reason")
        ).alias("dq_reason")
    )
)

In [0]:
silver_dq_orders = (
    silver_orders_df
    .join(
        order_dq_records,
        on="order_id",
        how="inner"
    )
)

In [0]:
display(
    silver_dq_orders
    .orderBy("order_id")
)

In [0]:
valid_orders = silver_orders_df.select("order_id").distinct()

silver_dq_payments = (
    silver_payments_df
    .join(
        valid_orders,
        on="order_id",
        how="left_anti"
    )
    .withColumn(
        "dq_reason",
        F.lit("INVALID_ORDER_ID")
    )
)

display(silver_dq_payments)

In [0]:
silver_dq_deliveries = (
    silver_deliveries_df
    .join(
        valid_orders,
        on="order_id",
        how="left_anti"
    )
    .withColumn(
        "dq_reason",
        F.lit("INVALID_ORDER_ID")
    )
)

display(silver_dq_deliveries)

In [0]:
silver_dq_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_dq_orders"
    )

silver_dq_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_dq_payments"
    )

silver_dq_deliveries.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_project.project.silver_dq_deliveries"
    )

In [0]:
%sql
SHOW TABLES IN ecommerce_project.project;